# Progetto Smart Vehicular Systems - Multi-modal black box recorder
  - Gabriele Arcese
  - Diego Colì

## Setup

In [ ]:
%pip install paho-mqtt

In [ ]:
import carla
import os
import time
import random
import socket
import threading
import queue
import numpy as np
import json
import paho.mqtt.client as mqtt
from dataclasses import dataclass, field, asdict
from typing import Optional
from collections import deque


try:
    import pygame
except ImportError:
    pygame = None
    
try:
    import cv2
    import numpy as np
except ImportError:
    cv2 = None
    np = None

Note: you may need to restart the kernel to use updated packages.
pygame 2.6.1 (SDL 2.28.4, Python 3.7.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Connessione al client

In [2]:
client = carla.Client("localhost", 2000)
client.set_timeout(15.0)
world = client.get_world()
spectator = world.get_spectator()

print(f"Client version: {client.get_client_version()}")
print(f"Server version: {client.get_server_version()}")

Client version: 0.9.15
Server version: 0.9.15


## MQTT client

In [ ]:
# queue thread-safe per accumulare messaggi ricevuti
mqtt_msgs_q = queue.Queue()
ring_buffer = deque(maxlen=600)  

def _on_connect(client, userdata, flags, rc):
    if rc == 0:
        client.connected_flag = True
        # sottoscrivi i topic desiderati
        client.subscribe([("recorder/control", 0), ("vehicle/+/status", 0)])
    else:
        client.connected_flag = False

def _on_message(client, userdata, msg):
    ts = time.time()
    payload = msg.payload
    # prova a decodificare json, fallback a stringa/bytes
    parsed = None
    try:
        parsed = json.loads(payload.decode("utf-8"))
    except Exception:
        try:
            parsed = payload.decode("utf-8")
        except Exception:
            parsed = payload  # raw bytes
    record = {"topic": msg.topic, "payload": parsed, "qos": msg.qos, "ts": ts}
    try:
        mqtt_msgs_q.put_nowait(record)
    except queue.Full:
        pass

def start_mqtt(broker="localhost", port=1883, keepalive=60):
    client = mqtt.Client()
    client.on_connect = _on_connect
    client.on_message = _on_message
    client.connected_flag = False
    client.connect(broker, port, keepalive=keepalive)
    client.loop_start()   # avvia loop in background thread
    return client

def stop_mqtt(client):
    if client is None: return
    try:
        client.loop_stop()
        client.disconnect()
    except Exception:
        pass

# Helper: svuota la queue e ritorna lista (da chiamare nel tick)
def drain_mqtt_queue():
    items = []
    while True:
        try:
            items.append(mqtt_msgs_q.get_nowait())
        except queue.Empty:
            break
    return items

## Dataclass EventRecord
Prodotto ad ogni tick della simulazione: è l'unità atomica del ring buffer.

| Campo | Sorgente CARLA | Tipo |
|---|---|---|
| `timestamp_sim` | `world.get_snapshot().timestamp.elapsed_seconds` | `float` |
| `timestamp_wall` | `time.time()` | `float` |
| `frame_id` | contatore incrementale | `int` |
| `camera_frames` | callback `sensor.camera.rgb` (una per camera) | `dict[str, np.ndarray]` |
| `location` | `vehicle.get_location()` | `carla.Location` |
| `velocity` | `vehicle.get_velocity()` | `carla.Vector3D` |
| `acceleration` | `vehicle.get_acceleration()` | `carla.Vector3D` |
| `control` | `vehicle.get_control()` | `carla.VehicleControl` |
| `warnings` | logica interna (LiDAR, collisione) | `list[str]` |
| `mqtt_messages` | thread MQTT | `list[str]` |
| `reasoning_text` | explanation agent | `str \| None` |
| `reasoning_ts` | wall-clock emissione agent | `float \| None` |

In [4]:
@dataclass
class VehicleState:
    """Stato cinematico e di controllo del veicolo in un istante."""
    x: float
    y: float
    z: float
    vx: float
    vy: float
    vz: float
    ax: float
    ay: float
    az: float
    throttle: float
    brake: float
    steer: float
    hand_brake: bool
    reverse: bool

@property
def speed_ms(self) -> float:
    """Velocità scalare in m/s."""
    return float(np.sqrt(self.vx**2 + self.vy**2 + self.vz**2))

@property
def speed_kmh(self) -> float:
    return self.speed_ms * 3.6

@dataclass
class EventRecord:
    """Unità atomica del ring buffer — un record per tick di simulazione."""
    # --- Temporali ---
    frame_id: int            # contatore tick, usato come chiave nel viewer
    timestamp_sim: float     # secondi simulazione (univoco per tick in sync mode)
    timestamp_wall: float    # wall-clock al momento della costruzione

    # --- Percezione ---
    vehicle_state: VehicleState

    # --- Camera (dict role → frame BGR; vuoto se nessun frame è ancora arrivato) ---
    # Esempio: {"front": np.ndarray, "rear": np.ndarray}
    camera_frames: dict = field(default_factory=dict, repr=False)

    # --- Logica e messaggistica ---
    warnings: list = field(default_factory=list)       # es. ["COLLISION_IMMINENT"]
    mqtt_messages: list = field(default_factory=list)  # payload raw ricevuti nel tick

    # --- Explanation agent ---
    reasoning_text: Optional[str] = None
    reasoning_ts: Optional[float] = None  # wall-clock di emissione dell'agente

def to_dict(self) -> dict:
    """Serializza tutto tranne camera_frames (salvati separatamente come JPEG)."""
    d = asdict(self)
    d.pop("camera_frames")  # i frame vanno in frames/<frame_id:06d>_<role>.jpg
    return d

def has_camera(self, role: str = None) -> bool:
    """True se almeno una camera (o la camera `role`) ha un frame nel tick."""
    if role:
        return role in self.camera_frames
    return len(self.camera_frames) > 0

def camera_roles(self) -> list:
    return list(self.camera_frames.keys())

def has_reasoning(self) -> bool:
    return self.reasoning_text is not None

Il **timestamp** di simulazione è la chiave di allineamento: tutto viene sincronizzato su di esso perché in modalità sincrona.

## VehicleState

In [5]:
# VehicleState partendo da un attore
def build_vehicle_state(vehicle) -> VehicleState:
    loc = vehicle.get_location()
    vel = vehicle.get_velocity()
    acc = vehicle.get_acceleration()
    ctrl = vehicle.get_control()
    return VehicleState(
    x=loc.x, y=loc.y, z=loc.z,
    vx=vel.x, vy=vel.y, vz=vel.z,
    ax=acc.x, ay=acc.y, az=acc.z,
    throttle=ctrl.throttle,
    brake=ctrl.brake,
    steer=ctrl.steer,
    hand_brake=ctrl.hand_brake,
    reverse=ctrl.reverse,
)

## Attivazione modalità sync
In modalità **sincrona**, il server avanza solo quando il client chiama `world.tick()`. Ciò è utile per esperimenti controllati e risultati riproducibili.


In [6]:
settings = world.get_settings()
settings.synchronous_mode = True
world.apply_settings(settings)
print("Modalità sincrona attivata")

Modalità sincrona attivata


## Spawn veicolo

In [7]:
def move_spectator_to(transform, spectator, distance=7.0, x=0, y=0, z=3.0, yaw=0, pitch=-15, roll=0):
    back_location = transform.location - transform.get_forward_vector() * distance
    
    back_location.x += x
    back_location.y += y
    back_location.z += z
    transform.rotation.yaw += yaw
    transform.rotation.pitch = pitch
    transform.rotation.roll = roll
    
    spectator_transform = carla.Transform(back_location, transform.rotation)
    
    spectator.set_transform(spectator_transform)

def spawn_vehicle(world, vehicle_index=16, spawn_index=10):
    blueprint_library = world.get_blueprint_library()
    vehicle_bp = blueprint_library.filter('vehicle.*')[vehicle_index]
    spawn_point = world.get_map().get_spawn_points()[spawn_index]
    vehicle = world.spawn_actor(vehicle_bp, spawn_point)
    return vehicle

def draw_on_screen(world, transform, content="O", color=carla.Color(0, 255, 0), life_time=20):
    world.debug.draw_string(transform.location, content, color=color, life_time=life_time)


In [8]:
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(True)

## Ciclo di simulazione

In [9]:
def tcp_check(host, port, timeout=2.0):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
        s.close()
        return True, None
    except Exception as e:
        return False, str(e)

targets = [("localhost", 2001), ("test.mosquitto.org", 1883)]
for host, port in targets:
    ok, err = tcp_check(host, port)
    print(f"{host}:{port} -> {'OK' if ok else 'FAIL'}{'' if ok else ': '+err}")

# Try connecting with paho to localhost (shows MQTT error if refused)
client = mqtt.Client()
try:
    client.connect("localhost", 2001, keepalive=5)
    client.loop_start()
    time.sleep(0.3)
    client.loop_stop()
    print("paho-mqtt: connection to localhost:2001 succeeded")
except Exception as e:
    print("paho-mqtt: connection to localhost:2001 failed:", e)
    print("\nHints:")
    print("- If you want a local broker, start Mosquitto (outside notebook) or run Docker:")
    print("  docker run -d --name mosquitto -p 2001:2001 -p 9001:9001 eclipse-mosquitto")
    print("- Or switch to the public test broker: test.mosquitto.org:1883")

localhost:2001 -> OK
test.mosquitto.org:1883 -> OK


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:17: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  app.launch_new_instance()


paho-mqtt: connection to localhost:2001 succeeded


## Collision trigger

In [ ]:
# --- Collision trigger + flusher setup ---
collision_event = threading.Event()
collision_info  = {}
flusher_stop    = threading.Event()

def _on_collision(event):
    other   = getattr(event, "other_actor", None)
    impulse = getattr(event, "normal_impulse", None)
    collision_info.clear()
    collision_info.update({
        "other_actor": str(other.type_id) if other else "unknown",
        "impulse":     str(impulse),
        "ts":          time.time(),
    })
    collision_event.set()

def flusher_thread(out_dir_base="recordings"):
    while not flusher_stop.is_set():
        if collision_event.wait(timeout=0.1):
            collision_event.clear()
            session_dir = f"{out_dir_base}_{time.strftime('%Y%m%dT%H%M%S')}"
            os.makedirs(session_dir, exist_ok=True)
            snapshot = list(ring_buffer)
            with open(os.path.join(session_dir, "events.jsonl"), "w", encoding="utf-8") as f:
                for rec in snapshot:
                    json.dump(rec.to_dict(), f, default=str)
                    f.write("\n")
            with open(os.path.join(session_dir, "metadata.json"), "w", encoding="utf-8") as f:
                json.dump({
                    "trigger":        "collision",
                    "collision_info": collision_info.copy(),
                    "n_frames":       len(snapshot),
                    "saved_at":       time.time(),
                }, f, indent=2, default=str)
            print(f"Flusher: saved {len(snapshot)} frames → {session_dir}")

## Avvio sensore collisione e flusher

In [ ]:
# --- Avvio sensore collisione e flusher ---
collision_sensor = None
try:
    col_bp = world.get_blueprint_library().find("sensor.other.collision")
    collision_sensor = world.spawn_actor(col_bp, carla.Transform(), attach_to=vehicle)
    collision_sensor.listen(_on_collision)
    flusher = threading.Thread(target=flusher_thread, args=("recordings",), daemon=True)
    flusher.start()
    print("Collision sensor e flusher avviati")
except Exception as e:
    print("Avvio collision sensor/flusher fallito:", e)

## Main simulation loop

In [ ]:
frame_id = 0
try:
    mqtt_client = start_mqtt(broker="localhost", port=2001)
    while True:
        world.tick()
        msgs = drain_mqtt_queue()
        record = EventRecord(
            frame_id=frame_id,
            timestamp_sim=world.get_snapshot().timestamp.elapsed_seconds,
            timestamp_wall=time.time(),
            vehicle_state=build_vehicle_state(vehicle),
            camera_frames={},
            warnings=[],
            mqtt_messages=msgs,
        )
        ring_buffer.append(record)
        frame_id += 1
        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0, pitch=-20)
        time.sleep(0.05)
except KeyboardInterrupt:
    stop_mqtt(mqtt_client)
    pass


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:31: DeprecationWarning: Callback API version 1 is deprecated, update to latest version


In [11]:
settings.synchronous_mode = False
world.apply_settings(settings)
print("Modalità sincrona disattivata")

Modalità sincrona disattivata


In [ ]:
# --- Teardown: flusher + collision sensor + veicolo ---
try:
    flusher_stop.set()
except NameError:
    pass

if "collision_sensor" in dir() and collision_sensor is not None:
    try:
        collision_sensor.stop()
    except Exception:
        pass
    try:
        collision_sensor.destroy()
    except Exception:
        pass
    collision_sensor = None

vehicle.destroy()
print("Veicolo distrutto")
vehicle = None


Veicolo distrutto
